# Step 001: Pfade und Datensatz-Setup

Dieses Notebook ist der portable Startpunkt vor `step01_training_lab.ipynb`.

Ziele:

- Projektroot automatisch finden oder per Umgebungsvariable setzen.
- Python-Interpreter fuer Trainingskommandos festlegen.
- Erwartete Ergebnisordner anlegen.
- WIDER-FACE-Datensatz pruefen.
- Optional lokale WIDER-FACE-ZIPs oder einen lokalen WIDER-FACE-Ordner uebernehmen.

Dieses Notebook startet kein Training.

## 1. Portable Pfade setzen

Die Suche funktioniert in zwei typischen Situationen:

- Notebook wird im Projektroot gestartet, der `face_model_lab/` enthaelt.
- Notebook wird direkt aus `face_model_lab/` gestartet.

Falls das nicht passt, setze vor dem Start `FACE_LAB_ROOT` auf den Projektroot.

In [6]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        lab = candidate / "face_model_lab"
        if (lab / "step00_common.py").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "step00_common.py").exists():
            return candidate.parent

    raise FileNotFoundError(
        "Projektroot nicht gefunden. Starte das Notebook im Projektordner "
        "oder setze FACE_LAB_ROOT."
    )

ROOT = Path(os.environ["FACE_LAB_ROOT"]).expanduser().resolve() if os.environ.get("FACE_LAB_ROOT") else find_project_root()
LAB = ROOT / "face_model_lab"
PYTHON = Path(os.environ.get("FACE_LAB_PYTHON", sys.executable)).expanduser().resolve()

if str(LAB) not in sys.path:
    sys.path.insert(0, str(LAB))

def show_command(parts):
    print(" ".join(str(p) for p in parts))

print("Projektroot:", ROOT)
print("Face Model Lab:", LAB)
print("Python:", PYTHON)
print("Notebook cwd:", Path.cwd().resolve())

Projektroot: /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt
Face Model Lab: /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/face_model_lab
Python: /usr/bin/python3.12
Notebook cwd: /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/face_model_lab


## 2. Projektordner und gemeinsame Pfade pruefen

Die Rohdaten liegen erwartet unter `ROOT/datasets/wider_face/`. Diese Zelle definiert die Pfade ohne Import von `step00_common.py`, damit der Setup-Check auch funktioniert, wenn Trainingsbibliotheken wie `torch` oder `cv2` noch fehlen.

In [7]:
DATASET_DIR = ROOT / "datasets" / "wider_face"
ANNOTATIONS_DIR = ROOT / "annotations"
MODEL_DIR = ROOT / "trained_models"
RESULTS_DIR = ROOT / "model_results"
YOLO_DATASET_DIR = ROOT / "datasets" / "wider_face_yolo"

def ensure_dirs():
    for directory in [ANNOTATIONS_DIR, MODEL_DIR, RESULTS_DIR, YOLO_DATASET_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

def wider_paths(split):
    if split == "train":
        image_dir = DATASET_DIR / "WIDER_train" / "WIDER_train" / "images"
        gt_file = DATASET_DIR / "wider_face_split" / "wider_face_split" / "wider_face_train_bbx_gt.txt"
    elif split == "val":
        image_dir = DATASET_DIR / "WIDER_val" / "WIDER_val" / "images"
        gt_file = DATASET_DIR / "wider_face_split" / "wider_face_split" / "wider_face_val_bbx_gt.txt"
    else:
        raise ValueError(f"Unknown split: {split}")
    return image_dir, gt_file

ensure_dirs()

print("DATASET_DIR:     ", DATASET_DIR)
print("ANNOTATIONS_DIR: ", ANNOTATIONS_DIR)
print("MODEL_DIR:       ", MODEL_DIR)
print("RESULTS_DIR:     ", RESULTS_DIR)
print("YOLO_DATASET_DIR:", YOLO_DATASET_DIR)

for path in [ANNOTATIONS_DIR, MODEL_DIR, RESULTS_DIR, YOLO_DATASET_DIR]:
    print(("OK      " if path.exists() else "FEHLT   ") + str(path))

DATASET_DIR:      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face
ANNOTATIONS_DIR:  /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/annotations
MODEL_DIR:        /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/trained_models
RESULTS_DIR:      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/model_results
YOLO_DATASET_DIR: /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face_yolo
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/annotations
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/trained_models
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/model_results
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face_yolo


## 3. Optional: Datensatz aus lokaler Quelle uebernehmen

Standardmaessig wird nichts kopiert oder entpackt. Setze `LOAD_DATASET = True`, wenn du lokale ZIP-Dateien oder einen bereits entpackten WIDER-FACE-Ordner uebernehmen willst.

Unterstuetzte lokale Quellen:

- Ordner mit `WIDER_train/`, `WIDER_val/`, `wider_face_split/`
- Ordner mit `WIDER_train.zip`, `WIDER_val.zip`, `wider_face_split.zip`

Quelle setzen ueber:

- `WIDER_FACE_SOURCE_DIR` als Umgebungsvariable, oder
- `SOURCE_DIR` direkt in der Zelle.

In [8]:
import shutil
import zipfile

LOAD_DATASET = False
SOURCE_DIR = Path(os.environ.get("WIDER_FACE_SOURCE_DIR", "")).expanduser()

# Beispiel, falls keine Umgebungsvariable genutzt wird:
# SOURCE_DIR = Path("/pfad/zu/wider_face_downloads")

def copy_dir_if_missing(src: Path, dst: Path) -> None:
    if dst.exists():
        print("OK      existiert bereits:", dst)
        return
    print("KOPIERE ", src, "->", dst)
    shutil.copytree(src, dst)

def extract_zip_if_needed(zip_path: Path, dst_root: Path) -> None:
    marker = dst_root / zip_path.stem
    if marker.exists():
        print("OK      bereits entpackt:", marker)
        return
    print("ENTPACKE", zip_path, "->", dst_root)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(dst_root)

if LOAD_DATASET:
    if not SOURCE_DIR.exists():
        raise FileNotFoundError(f"SOURCE_DIR existiert nicht: {SOURCE_DIR}")

    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    expected_dirs = ["WIDER_train", "WIDER_val", "wider_face_split"]
    for name in expected_dirs:
        src_dir = SOURCE_DIR / name
        src_zip = SOURCE_DIR / f"{name}.zip"
        dst_dir = DATASET_DIR / name

        if src_dir.exists():
            copy_dir_if_missing(src_dir, dst_dir)
        elif src_zip.exists():
            extract_zip_if_needed(src_zip, DATASET_DIR)
        else:
            print("FEHLT   Quelle fuer", name, "in", SOURCE_DIR)
else:
    print("LOAD_DATASET=False: Es wird nichts kopiert oder entpackt.")

LOAD_DATASET=False: Es wird nichts kopiert oder entpackt.


## 4. Optional: Download-Vorbereitung

Automatischer Download ist deaktiviert, weil viele Notebook-Umgebungen offline laufen oder WIDER FACE je nach Quelle andere Download-Links nutzt. Wenn du stabile lokale/Uni-Links hast, kannst du sie unten eintragen und `DOWNLOAD_DATASET = True` setzen.

In [9]:
from urllib.request import urlretrieve

DOWNLOAD_DATASET = False
DOWNLOAD_DIR = ROOT / "downloads" / "wider_face"

DATASET_URLS = {
    # "WIDER_train.zip": "https://.../WIDER_train.zip",
    # "WIDER_val.zip": "https://.../WIDER_val.zip",
    # "wider_face_split.zip": "https://.../wider_face_split.zip",
}

if DOWNLOAD_DATASET:
    if not DATASET_URLS:
        raise ValueError("DATASET_URLS ist leer. Trage zuerst Download-URLs ein.")
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    for filename, url in DATASET_URLS.items():
        target = DOWNLOAD_DIR / filename
        if target.exists():
            print("OK      Download existiert:", target)
            continue
        print("DOWNLOAD", url, "->", target)
        urlretrieve(url, target)
else:
    print("DOWNLOAD_DATASET=False: Kein Download.")

DOWNLOAD_DATASET=False: Kein Download.


## 5. Datensatzstruktur pruefen

Diese Zelle prueft, ob die Pfade existieren, die `step00_common.wider_paths()` fuer Training und Validierung verwendet.

In [10]:
required_paths = []
for split in ["train", "val"]:
    image_dir, gt_file = wider_paths(split)
    required_paths.extend([image_dir, gt_file])

missing = [path for path in required_paths if not path.exists()]
STRICT_DATASET_CHECK = False

print("Dataset dir:", DATASET_DIR)
for path in required_paths:
    print(("OK      " if path.exists() else "FEHLT   ") + str(path))

DATASET_READY = not missing

if missing:
    print("\nWARNUNG: WIDER-FACE-Daten fehlen oder liegen an anderer Stelle.")
    print("Setze LOAD_DATASET=True mit SOURCE_DIR oder lege die Daten unter DATASET_DIR ab.")

if missing and STRICT_DATASET_CHECK:
    raise FileNotFoundError(
        "WIDER-FACE-Daten fehlen oder liegen an anderer Stelle:\n"
        + "\n".join(str(path) for path in missing)
    )

Dataset dir: /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face/WIDER_train/WIDER_train/images
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face/wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face/WIDER_val/WIDER_val/images
OK      /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/datasets/wider_face/wider_face_split/wider_face_split/wider_face_val_bbx_gt.txt


## 6. Annotationen inhaltlich pruefen

Wenn diese Zelle laeuft, sind die Ground-Truth-Dateien lesbar und enthalten plausible Gesichtsannotationen.

In [11]:
import statistics

def parse_wider_face_gt(gt_file):
    lines = gt_file.read_text(encoding="utf-8", errors="replace").splitlines()
    annotations = {}
    cursor = 0

    while cursor < len(lines):
        rel_path = lines[cursor].strip()
        cursor += 1
        if not rel_path or cursor >= len(lines):
            continue

        try:
            face_count = int(lines[cursor].strip())
        except ValueError:
            break
        cursor += 1

        if face_count == 0 and cursor < len(lines):
            parts = lines[cursor].strip().split()
            if len(parts) >= 4:
                try:
                    [float(value) for value in parts[:4]]
                    cursor += 1
                except ValueError:
                    pass

        faces = []
        for _ in range(face_count):
            if cursor >= len(lines):
                break
            parts = lines[cursor].strip().split()
            cursor += 1
            if len(parts) < 4:
                continue
            x, y, w, h = map(float, parts[:4])
            if w > 0 and h > 0:
                faces.append({"bbox": [x, y, w, h]})
        annotations[rel_path] = faces
    return annotations

def dataset_summary(split):
    image_dir, gt_file = wider_paths(split)
    annotations = parse_wider_face_gt(gt_file)
    face_counts = [len(faces) for faces in annotations.values()]
    non_empty = [count for count in face_counts if count > 0]
    return {
        "split": split,
        "image_dir": str(image_dir),
        "gt_file": str(gt_file),
        "images": len(face_counts),
        "images_with_faces": len(non_empty),
        "images_without_faces": len(face_counts) - len(non_empty),
        "faces": sum(face_counts),
        "faces_per_image_mean": statistics.fmean(face_counts) if face_counts else 0.0,
        "faces_per_face_image_mean": statistics.fmean(non_empty) if non_empty else 0.0,
        "faces_per_image_median": statistics.median(face_counts) if face_counts else 0.0,
        "faces_per_image_max": max(face_counts) if face_counts else 0,
    }

if not globals().get("DATASET_READY", False):
    print("Datensatz ist noch nicht vollstaendig vorhanden. Plausibilitaetscheck wird uebersprungen.")
    summary = []
else:
    summary = [dataset_summary("train"), dataset_summary("val")]

for row in summary:
    print(
        f"{row['split']}: "
        f"images={row['images']}, "
        f"faces={row['faces']}, "
        f"images_with_faces={row['images_with_faces']}, "
        f"mean_faces/image={row['faces_per_image_mean']:.2f}, "
        f"max_faces/image={row['faces_per_image_max']}"
    )

if summary and any(row["images"] == 0 for row in summary):
    raise RuntimeError("Mindestens ein Split enthaelt keine Bilder laut Annotationen.")

if summary and any(row["faces"] == 0 for row in summary):
    raise RuntimeError("Mindestens ein Split enthaelt keine Gesichter laut Annotationen.")

train: images=12880, faces=159393, images_with_faces=12876, mean_faces/image=12.38, max_faces/image=1968
val: images=3226, faces=39697, images_with_faces=3222, mean_faces/image=12.31, max_faces/image=709


## 7. Trainingskommandos testen, ohne Training zu starten

Diese Zelle erzeugt nur ein Beispielkommando. Wenn Pfade und Python korrekt aussehen, kann danach `step01_training_lab.ipynb` genutzt werden.

In [12]:
BATCH_SIZE = 2
IMGSZ = 768
WORKERS = 0

SMOKE_REDUCTION = 20
SMOKE_EPOCHS = 1

example_cmd = [
    PYTHON, LAB / "step02_train_ultralytics_detector.py",
    "--family", "yolo", "--base", "yolov8m.pt",
    "--epochs", SMOKE_EPOCHS, "--batch", BATCH_SIZE, "--reduction", SMOKE_REDUCTION,
    "--imgsz", IMGSZ, "--train-limit", 0, "--val-limit", 0, "--workers", WORKERS,
]

print("Beispielkommando ausfuehren aus ROOT:")
print("cd", ROOT)
show_command(example_cmd)

Beispielkommando ausfuehren aus ROOT:
cd /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt
/usr/bin/python3.12 /home/clemens/OneDrive/Dokumente/Master MIM/Fächer/Muserterkennung/Projekt/face_model_lab/step02_train_ultralytics_detector.py --family yolo --base yolov8m.pt --epochs 1 --batch 2 --reduction 20 --imgsz 768 --train-limit 0 --val-limit 0 --workers 0
